# Ipotesi 1 - test dell'oracolo - VideoLLaMA 2 7B

Ogni domanda ha 6 opzioni: le 5 originali, piu' un **oracolo**, cioe' un ingrediente
molto superiore agli altri per il nutriente chiesto, ma **non presente nella ricetta del video**.

Due condizioni, stessa domanda e stesso prompt:

| Condizione | Cosa vede il modello | Cosa ci aspettiamo se guarda il video |
|---|---|---|
| `O0_video` | il video + le 6 opzioni | sceglie l'oracolo raramente |
| `O1_testo` | solo le 6 opzioni, nessun video | sceglie l'oracolo spesso (e' il "caso" del testo) |




**Runtime > GPU L4.** 


In [ ]:
# Runtime: Runtime > Cambia tipo di runtime > GPU L4
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della VRAM

!nvidia-smi
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Nessuna GPU: Runtime > Cambia tipo di runtime > L4"
print("GPU:", gpu, "| Python:", sys.version.split()[0])
if "L4" not in gpu:
    print("ATTENZIONE: il notebook e' tarato su L4. Un'altra GPU va bene se ha almeno 24 GB.")
# Serve Python 3.12, come il baseline: su 3.13 tokenizers 0.19.1 non ha la wheel e la
# compilazione con Rust fallisce, quindi pip non installa transformers 4.40 ne' decord.
assert sys.version_info[:2] == (3, 12), (
    f"Python {sys.version.split()[0]}: serve 3.12 (come il baseline). Runtime > Cambia tipo di runtime > "
    "Versione runtime: sceglierne una con Python 3.12, poi riavviare la sessione.")

Sat Nov 21 12:14:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# PyTorch 2.7.0, come il baseline. torchaudio va allineato: quello preinstallato da Colab
# e' compilato per un altro torch e non si potrebbe importare.
%pip install -q torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 216.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 47.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 49.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 68.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 90.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 61.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 63.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 58.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 64.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 67.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Repo ufficiale di VideoLLaMA 2
import os, shutil, sys
VL2_DIR = "/content/VideoLLaMA2"
VL2_COMMIT = "c0bb03abf6b8a6b9a8dccac006fb4db5d4d9e414"   # ultimo commit del repo (gennaio 2025)

%cd /content
if os.path.exists(VL2_DIR):
    shutil.rmtree(VL2_DIR)          # toglie anche le patch di una sessione precedente
!git clone -q https://github.com/DAMO-NLP-SG/VideoLLaMA2.git {VL2_DIR}
!git -C {VL2_DIR} checkout -q {VL2_COMMIT}
!git -C {VL2_DIR} log -1 --format="VideoLLaMA2: commit %h del %cd"

if VL2_DIR not in sys.path:
    sys.path.insert(0, VL2_DIR)

/content
VideoLLaMA2: commit c0bb03a del Thu Jan 23 12:20:22 2025 +0800


In [ ]:
%cd /content/VideoLLaMA2

/content/VideoLLaMA2


In [ ]:
%pip install -q --upgrade pip setuptools wheel   # servono a compilare tokenizers se Python e' 3.13

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 60.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
# Versioni ufficiali di VideoLLaMA2 (pyproject.toml del repo), le stesse del baseline:
#   transformers==4.40.0, tokenizers==0.19.1, accelerate==0.26.1, decord==0.6.0
# - tokenizers 0.19.1 ha la wheel per Python 3.12 ma non per 3.13: li' si compila con Rust (alcuni minuti).
# - huggingface_hub < 1.0 e' richiesto da transformers 4.40 (Colab ha la 1.x).
# - timm < 1.1: il repo importa timm.models.layers, deprecato.
# - peft NON si installa: VideoLLaMA2 non lo usa in inferenza, e una versione nuova
#   potrebbe trascinare un aggiornamento di transformers.
# Gli avvisi rossi di pip su gradio, sentence-transformers, peft o torchao sono attesi:
# sono pacchetti di Colab che questo notebook non importa.
#
# %pip non si ferma se l'installazione fallisce: se un pacchetto (es. tokenizers) non si
# compila, pip annulla TUTTO il comando e restano le versioni preinstallate di Colab.
# Per questo si usa pip() qui sotto, che si ferma e mostra l'errore vero.
import sys, os, subprocess

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join((r.stdout + r.stderr).splitlines()[-60:]))
        raise RuntimeError(f"pip install fallito: {' '.join(args)} (errore qui sopra)")

print("Python:", sys.version)

if sys.version_info >= (3, 13):
    print("Python 3.13: compilo tokenizers 0.19.1 con Rust (alcuni minuti)...")
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
    os.environ["PATH"] += ":/root/.cargo/bin"
    os.environ["PYO3_USE_ABI3_FORWARD_COMPATIBILITY"] = "1"   # pyo3 di tokenizers 0.19 non conosce ancora 3.13
    pip("tokenizers==0.19.1", "--no-binary", "tokenizers")
else:
    pip("tokenizers==0.19.1")

pip("transformers==4.40.0", "tokenizers==0.19.1", "accelerate==0.26.1", "huggingface_hub>=0.23,<1.0",
    "timm>=1.0.3,<1.1", "einops", "decord==0.6.0", "av", "imageio", "opencv-python")

# Controllo: se le versioni non sono quelle del baseline ci si ferma qui
from importlib.metadata import version, PackageNotFoundError
def _v(k):
    try:
        return version(k)
    except PackageNotFoundError:
        return "NON INSTALLATO"
_att = {"torch": "2.7.0", "transformers": "4.40.0", "tokenizers": "0.19.1",
        "accelerate": "0.26.1", "decord": "0.6.0"}
_vers = {k: _v(k) for k in _att}
print(_vers)
_err = {k: v for k, v in _vers.items() if not v.startswith(_att[k])}
assert not _err, f"versioni errate {_err}, attese {_att}: controllare l'output di pip sopra"

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
{'torch': '2.7.0+cu128', 'transformers': '4.40.0', 'tokenizers': '0.19.1', 'accelerate': '0.26.1', 'decord': '0.6.0'}


In [ ]:
import os
# Evita il lookup del secret HF_TOKEN di Colab che va in timeout e "appende" il download
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # download piu' stabile
os.environ.pop("HF_TOKEN", None)

import gc, sys, pathlib, torch
# Se torch era gia' stato importato prima dell'installazione, in memoria c'e' ancora quello vecchio
assert torch.__version__.startswith("2.7.0"), (
    f"torch {torch.__version__} in memoria, atteso 2.7.0: Runtime > Riavvia sessione, poi rieseguire dall'inizio")
assert torch.cuda.is_available(), "CUDA non disponibile: il runtime non ha la GPU"

# Libero la VRAM da tentativi precedenti
for name in ['model', 'processor', 'tokenizer']:
    if name in dir():
        exec(f"del {name}")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM libera: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

# -----------------------------------------------------------------------------
# PATCH encoder.py: il repo forza a mano config._attn_implementation="flash_attention_2"
# sulla CLIP/Siglip vision tower, ma questi modelli non supportano flash-attn -> ValueError.
# Sostituiamo con "eager" DIRETTAMENTE nel file sorgente, prima di importare videollama2.
# -----------------------------------------------------------------------------
_enc = pathlib.Path("/content/VideoLLaMA2/videollama2/model/encoder.py")
_txt = _enc.read_text()
if '"flash_attention_2"' in _txt:
    _enc.write_text(_txt.replace('"flash_attention_2"', '"eager"'))
    print("Patch applicata: flash_attention_2 -> eager in encoder.py")
else:
    print("Nessuna patch necessaria (flash_attention_2 non trovato in encoder.py)")

# Svuoto la cache dei moduli gia' importati, cosi' l'import qui sotto rilegge il file patchato
for _m in list(sys.modules):
    if _m.startswith("videollama2"):
        del sys.modules[_m]

# 1) Scarico lo snapshot VideoLLaMA2-7B (circa 16 GB)
from huggingface_hub import snapshot_download
model_path = snapshot_download("DAMO-NLP-SG/VideoLLaMA2-7B")
print("Modello scaricato in:", model_path)

# 2) Inizializzo passando il path LOCALE (niente altre chiamate di rete).
from videollama2 import model_init as videollama2_model_init

device = "cuda:0"
print("Inizio caricamento modello...")
model, processor, tokenizer = videollama2_model_init(
    model_path,
    device_map={"": device},
    attn_implementation="eager",   # forza eager sul modello linguistico
    use_flash_attn=False,
)

# imposto il modello in modalita' inferenza
model.eval()
print("Modello VideoLLaMA2-7B caricato con successo - dtype:", next(model.parameters()).dtype)
print("num_frames usati dal processor:", getattr(model.config, "num_frames", "default (8)"))
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1024**3:.1f} GiB "
      f"su {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

VRAM libera: 21.84 GiB
Patch applicata: flash_attention_2 -> eager in encoder.py


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

... (16 outputs left out)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Modello VideoLLaMA2-7B caricato con successo - dtype: torch.float16
num_frames usati dal processor: default (8)
VRAM occupata: 15.5 GiB su 22.0 GiB


## 2. Percorsi e indice dei video

Il notebook legge codice e dati dalla cartella del progetto su Drive (`src/`, `data/`),
copiati dal Mac con `scripts/sync_drive.sh`. I risultati vanno in `results/oracolo/`.

In [ ]:
# Codice e dati arrivano su Drive con scripts/sync_drive.sh, lanciato dal Mac prima di questa cella.
import os, sys, glob, importlib
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
SRC = os.path.join(BASE, "src")
assert os.path.exists(os.path.join(SRC, "hdepic", "oracolo.py")), (
    f"Manca {SRC}/hdepic/oracolo.py su Drive: lanciare scripts/sync_drive.sh dal Mac")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from hdepic import eval as H
from hdepic import oracolo as A
importlib.reload(H); importlib.reload(A)         # se la cella viene rieseguita

# file che devono essere su Drive prima di iniziare (i percorsi li ricava src/hdepic/oracolo.py)
RICHIESTI = [A.ORACOLI, A.DOMANDE_ORIGINALI, A.ANNOTAZIONI_ORACOLO]
mancanti = [p for p in RICHIESTI if not os.path.exists(p)]
assert not mancanti, f"Mancano su Drive: {mancanti}. Lanciare scripts/sync_drive.sh dal Mac"
os.makedirs(A.RISULTATI, exist_ok=True)

MODELLO = "videollama2"
VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
assert VIDEO_ROOTS, f"Nessuna cartella video* in {BASE}"
VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
READABLE = {}     # cache: aprire i video da Drive e' lento, si controlla una volta sola

print("cartelle video:", [os.path.basename(d) for d in VIDEO_ROOTS])
print(f"video indicizzati: {len(VIDEO_INDEX['sampled'])} campionati, {len(VIDEO_INDEX['full'])} integrali")
print("annotazioni a 6 opzioni presenti:", os.path.exists(A.ANNOTAZIONI_ORACOLO))
print("baseline C0 presente:", os.path.exists(os.path.join(A.RISULTATI, A.nome_file(MODELLO, "C0_video"))))

Mounted at /content/drive
cartelle video: ['Videos Nutrition Change Sampled', 'Videos Nutrition Estimation Sampled', 'videos_image_nutrition_estimation_sampled']
video indicizzati: 50 campionati, 0 integrali
annotazioni a 6 opzioni presenti: True
baseline C0 presente: True


## 3. Controllo dei file

Ogni domanda deve avere un video leggibile. Le domande con file rotti vengono escluse e
risultano nella tabella con il loro N.

In [ ]:
cov, dettagli = H.preflight(A.ANN_ORACOLO_DIR, VIDEO_INDEX, tasks=[A.TASK], cache=READABLE)
print(cov.to_string(index=False))
rotti = sorted({v for ids in dettagli[A.TASK].values() for v in ids})
if rotti:
    print("file mancanti o non decodificabili:", rotti)
utili = int(cov.loc[cov["task"] == A.TASK, "utilizzabili"].iloc[0])
assert utili > 0, "Nessuna domanda con un video leggibile: controllare le cartelle video su Drive"

                      task input  domande  utilizzabili  scartate file_inutilizzabili
video_nutrition_estimation video       49            49         0                   0
                    TOTALE             49            49         0                    


## 4. Funzioni di inferenza

- **`infer_video`** e' la stessa `videollama2_infer` del baseline (usata per O0 e per il controllo C0).
- **`infer_testo`** usa la modalita' `text` di `mm_infer`: nessun tensore visivo, nessun token video.
  Il resto (decoding greedy, `max_new_tokens`) e' identico.

In [ ]:
import numpy as np
from videollama2 import mm_infer

MAX_NEW_TOKENS = 32          # basta: la risposta attesa e' una lettera
USE_SYSTEM_PROMPT = None     # il paper non usa Systemp Prompt (se lo utilizzassimo le performance si alzano)
NUM_FRAMES = getattr(model.config, "num_frames", 8)   # elementi visivi per domanda (default 8)

def sottocampiona(paths, n=NUM_FRAMES):
    """Riduce una lista di immagini a n elementi uniformi (come _immagini in LLaVA).

    Serve perche' process_video ricampiona a num_frames SOLO quando riceve il path di un
    video: una lista di immagini la carica tutta e la tronca a MAX_FRAMES=32. Senza questo
    passaggio i 32 pannelli arriverebbero tutti al modello (4x i token visivi del baseline).
    Liste piu' corte di n (es. le 5 immagini di image_nutrition_estimation) restano invariate.
    """
    if len(paths) <= n:
        return paths
    sel = np.linspace(0, len(paths) - 1, n, dtype=int)
    return [paths[i] for i in sel]

def videollama2_infer(visual, prompt):
    """visual: path di un mp4, oppure lista di path di immagini (frame o pannelli)."""
    if isinstance(visual, list):
        visual = sottocampiona(visual)
    tensor = processor["video"](visual)
    # stesso numero di elementi visivi in ogni condizione (video, pannelli, immagini)
    assert tensor.shape[0] == NUM_FRAMES, f"attesi {NUM_FRAMES} elementi visivi, arrivati {tensor.shape[0]}"
    return mm_infer(tensor, prompt, model, tokenizer, modal="video",
                    do_sample=False,          # greedy: run deterministico e ripetibile
                    max_new_tokens=MAX_NEW_TOKENS)

print("num_frames usati dal processor:", NUM_FRAMES)
print("dtype:", next(model.parameters()).dtype)

num_frames usati dal processor: 8
dtype: torch.float16


In [ ]:
def infer_testo(prompt):
    """O1: solo testo, nessun input visivo."""
    return mm_infer(None, prompt, model, tokenizer, modal="text",
                    do_sample=False, max_new_tokens=MAX_NEW_TOKENS)

infer_video = videollama2_infer
INFER = {"O0_video": infer_video, "O1_testo": A.solo_testo(infer_testo)}
print("condizioni pronte:", list(INFER))

condizioni pronte: ['O0_video', 'O1_testo']


## 6. Controllo Baseline

Rifa' 5 domande del baseline e le confronta con quelle salvate. Devono coincidere tutte.


In [ ]:
BASELINE_C0 = os.path.join(A.RISULTATI, A.nome_file(MODELLO, "C0_video"))
if not os.path.exists(BASELINE_C0):
    print(f"Controllo C0 SALTATO: manca {BASELINE_C0}.\n"
          "Copiare li' il .jsonl del baseline di video_nutrition_estimation e rilanciare questa cella.")
else:
    ctrl = A.esegui(MODELLO, "C0_controllo", infer_video, A.ANN_ORIGINALI_DIR, VIDEO_INDEX,
                    limit=5, readable_cache=READABLE)
    A.controllo_c0(os.path.join(A.RISULTATI, A.nome_file(MODELLO, "C0_controllo")), BASELINE_C0)

  50 domande utilizzabili


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  [1/50] OK    0 | pred=3 gt=3 | leading_letter     | 'D'
  [2/50] KO    1 | pred=2 gt=0 | leading_letter     | 'C'
  [3/50] OK    2 | pred=4 gt=4 | leading_letter     | 'E'
  [4/50] OK    3 | pred=4 gt=4 | leading_letter     | 'E'
  [5/50] OK    4 | pred=1 gt=1 | leading_letter     | 'B'
Controllo C0: 5/5 risposte identiche al baseline


## 7. Run completi

In [ ]:
for c in INFER:
    df = A.esegui(MODELLO, c, INFER[c], A.ANN_ORACOLO_DIR, VIDEO_INDEX, readable_cache=READABLE)
    print(f"{c}: {len(df)} domande")

  49 domande utilizzabili
  [1/49] OK    0 | pred=3 gt=3 | leading_letter     | 'D'
  [2/49] KO    1 | pred=3 gt=0 | leading_letter     | 'D'
  [3/49] OK   10 | pred=3 gt=3 | leading_letter     | 'D'
  [4/49] OK   11 | pred=5 gt=5 | leading_letter     | 'F'
  [5/49] OK   12 | pred=1 gt=1 | leading_letter     | 'B'
  [6/49] KO   13 | pred=1 gt=0 | leading_letter     | 'B'
  [7/49] KO   14 | pred=5 gt=0 | leading_letter     | 'F'
  [8/49] OK   15 | pred=5 gt=5 | leading_letter     | 'F'
  [9/49] KO   16 | pred=0 gt=1 | leading_letter     | 'A'
  [10/49] KO   17 | pred=2 gt=1 | leading_letter     | 'C'
  [11/49] KO   18 | pred=2 gt=4 | leading_letter     | 'C'
  [12/49] OK   19 | pred=0 gt=0 | leading_letter     | 'A'
  [13/49] OK    2 | pred=5 gt=5 | leading_letter     | 'F'
  [14/49] OK   20 | pred=2 gt=2 | leading_letter     | 'C'
  [15/49] KO   21 | pred=2 gt=0 | leading_letter     | 'C'
  [16/49] KO   22 | pred=0 gt=5 | leading_letter     | 'A'
  [17/49] KO   23 | pred=1 gt=3 | leadi

## 8. Prima lettura
La scelta dell'oracolo e il verdetto vanno letti nel notebook `03_analisi`.

In [ ]:
A.confronta([os.path.join(A.RISULTATI, A.nome_file(MODELLO, c)) for c in INFER])

run                                                       n  oracolo      %           IC 95%  accur.
----------------------------------------------------------------------------------------------------
videollama2_O0_video_video_nutrition_estimation.jsonl    49       16  32.7%           21-47%   38.8%
videollama2_O1_testo_video_nutrition_estimation.jsonl    49       14  28.6%           18-42%   40.8%

Caso: 16.7%   soglia per 'risponde dal testo': 50%
